In [ ]:
import pandas as pd
from collections import Counter
import numpy as np
from pathlib import Path
from scipy.stats import entropy  # entropy is a function to compute KL-divergence

In [ ]:
#Define QID - question matching -- 2020 ANES
qids = ["V201324","V201416", "V202234", "V202257", "V202287", "V202332", "V202337", "V202348", "V202371", "V202378"]
name_dict = {"V201324" : "Current Economy", "V201416" : "Gay Marriage", "V202234" : "Refugee Allowing", 
            "V202257" : "Income Inequality", "V202287" : "Gender Role", "V202332": "Climate Change",
            "V202337": "Gun Regulation", "V202348": "Drug Addiction", "V202371": "Race Diversity", "V202378": "Health Insurance"}

def get_counts(path, qids): #function to get a dict with counts for each question
    value_list = []

    for id in qids:
        df = pd.read_csv(path + id + ".csv")
        df['Response'] = pd.to_numeric(df['Response'], errors='coerce') 
        value_counts = Counter(df['Response'].dropna())

        if id in ["V201324", "V202332"]:
            answers = {i: value_counts[i] for i in range(1, 6)}
        else:
            answers = {i: value_counts[i] for i in range(1, 4)}

        value_list.append({name_dict[id]: answers})

    return value_list


### Select model

In [ ]:
# LLama8b
model_name = 'llama8b'
replicate_path = "../Publication Results/8B-Llama/main_mq_results/full_results_2020_"
reformulated_path = "../Publication Results/8B-Llama/main_mq_reform_3rdP_results/full_results_2020_"
priming_path = "../Publication Results/8B-Llama/Priming/Original/full_results_2020_"
preamble_path = "../Publication Results/8B-Llama/Preamble/Original/full_results_2020_"
reversed_path = "../Publication Results/8B-Llama/main_mq_reverseV2_results/full_results_2020_"



In [ ]:
# LLama70b
model_name = 'llama70b'
replicate_path = "../Publication Results/70B-Llama/main_mq_results/full_results_2020_"
reformulated_path = "../Publication Results/70B-Llama/main_mq_reform_results/full_results_2020_"
priming_path = "../Publication Results/70B-Llama/main_mq_priming_results/full_results_2020_"
preamble_path = "../Publication Results/70B-Llama/main_mq_preamble_results/full_results_2020_"
reversed_path = "../Publication Results/70B-Llama/main_mq_reverseV2_results/full_results_2020_"
 

In [ ]:
# GPT-4.1
model_name = 'gpt4.1'
# T=0
replicate_path = "../Publication Results/Gpt4.1-Mini-Determinstic/main_mq_results/full_results_2020_"
reformulated_path = "../Publication Results/Gpt4.1-Mini-Determinstic/main_mq_reform_results/full_results_2020_"
priming_path = "../Publication Results/Gpt4.1-Mini-Determinstic/main_mq_priming_results/full_results_2020_"
preamble_path = "../Publication Results/Gpt4.1-Mini-Determinstic/main_mq_preamble_results/full_results_2020_"
reversed_path = "../Publication Results/Gpt4.1-Mini-Determinstic/main_mq_reverseV2_results/full_results_2020_"

 


In [ ]:
replicate_results = get_counts(replicate_path, qids)
reformulated_results = get_counts(reformulated_path, qids)
priming_results = get_counts(priming_path, qids)
preamble_results = get_counts(preamble_path, qids)

#Only some IDs were used for reverse-coded
qids_reverse_coded = ["V201416", "V202234", "V202257", "V202287", "V202332", "V202371"]
reversed_results = get_counts(reversed_path, qids_reverse_coded)

In [ ]:
def get_human_counts(path, qids): #getting results from anes 2020
     anes_df = pd.read_csv(path)
     human_values = []

     for id in qids:
          human_counts = Counter(anes_df[id].dropna())
          if id in ["V201324", "V202332"]:
               answers = {i: human_counts[i] for i in range(1, 6)}
          else:
               answers = {i: human_counts[i] for i in range(1, 4)}

          human_values.append({name_dict[id]: answers})

     return human_values

human_results = get_human_counts("../data/2020 ANES_test.csv", qids)

/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_57581/1072730952.py:2: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  anes_df = pd.read_csv(path)


In [451]:
from copy import deepcopy 

reversed_results = {list(d.keys())[0]: list(d.values())[0] for d in reversed_results}
reversed_results_copy = deepcopy(reversed_results)

reversed_results["Gay Marriage"][1] = reversed_results_copy["Gay Marriage"][3]
reversed_results["Gay Marriage"][3] = reversed_results_copy["Gay Marriage"][1]

reversed_results["Refugee Allowing"][1] = reversed_results_copy["Refugee Allowing"][2]
reversed_results["Refugee Allowing"][2] = reversed_results_copy["Refugee Allowing"][1]

reversed_results["Income Inequality"][1] = reversed_results_copy["Income Inequality"][2]
reversed_results["Income Inequality"][2] = reversed_results_copy["Income Inequality"][1]

reversed_results["Gender Role"][1] = reversed_results_copy["Gender Role"][2]
reversed_results["Gender Role"][2] = reversed_results_copy["Gender Role"][1]

reversed_results["Climate Change"][1] = reversed_results_copy["Climate Change"][5]
reversed_results["Climate Change"][2] = reversed_results_copy["Climate Change"][4]
reversed_results["Climate Change"][4] = reversed_results_copy["Climate Change"][2]
reversed_results["Climate Change"][5] = reversed_results_copy["Climate Change"][1]

reversed_results["Race Diversity"][1] = reversed_results_copy["Race Diversity"][2]
reversed_results["Race Diversity"][2] = reversed_results_copy["Race Diversity"][1]

reversed_results = [{key: value} for key, value in reversed_results.items()]

In [452]:
#JSD divergence is a more numerically stable version of KL-divergence - better for our case
def jsd_divergence_between_sets(set1, set2, base=2):
    jsd_results = {}

    dict1 = {list(d.keys())[0]: list(d.values())[0] for d in set1}
    dict2 = {list(d.keys())[0]: list(d.values())[0] for d in set2}

    for question in dict1.keys():
        if question not in dict2:
            continue 

        p_counts = dict1[question]
        q_counts = dict2[question]

        p = np.array([p_counts.get(k, 0) for k in p_counts], dtype=float)
        q = np.array([q_counts.get(k, 0) for k in p_counts], dtype=float)

        p /= p.sum()
        q /= q.sum()

        epsilon = 1e-10
        p = np.clip(p, epsilon, 1)
        q = np.clip(q, epsilon, 1)

        m = 0.5 * (p + q)
        jsd_pq = 0.5 * (entropy(p, m, base=base) + entropy(q, m, base=base)) 
        jsd_results[question] = jsd_pq

    return jsd_results

In [ ]:
#Get all the results and divergences
jsd_results_replicate = jsd_divergence_between_sets(human_results, replicate_results, base=2)
jsd_results_reformulated = jsd_divergence_between_sets(human_results, reformulated_results, base=2)
jsd_results_priming = jsd_divergence_between_sets(human_results, priming_results, base=2)
jsd_results_preamble = jsd_divergence_between_sets(human_results, preamble_results, base=2)
jsd_results_reversed = jsd_divergence_between_sets(human_results, reversed_results, base = 2)


diff_reformulated_vs_replicate = {key: (jsd_results_reformulated[key] - jsd_results_replicate[key]) for key in jsd_results_replicate.keys()}
diff_priming_vs_replicate = {key: (jsd_results_priming[key] - jsd_results_replicate[key]) for key in jsd_results_replicate.keys()}
diff_preamble_vs_replicate = {key: (jsd_results_preamble[key] - jsd_results_replicate[key]) for key in jsd_results_replicate.keys()}
diff_reversed_vs_replicate = {key: (jsd_results_reversed[key] - jsd_results_replicate[key]) for key in jsd_results_reversed.keys()}


In [454]:
# format data
# Difference in JS-divergence between replicate and 4 experimental conditions
results_mapping = [
    ('Replicate', jsd_results_replicate),
    ('Reformulated - Replicate', diff_reformulated_vs_replicate),
    ('Reverse-coded - Replicate', diff_reversed_vs_replicate),
    ('Priming - Replicate', diff_priming_vs_replicate),
    ('Preamble - Replicate', diff_preamble_vs_replicate),
]

question_order = ['Climate Change','Current Economy','Refugee Allowing','Health Insurance','Income Inequality','Race Diversity','Drug Addiction','Gay Marriage','Gun Regulation','Gender Role']
sig_df = pd.DataFrame({'Question': question_order})
for column_name, result_dict in results_mapping:
    # Extract values from the dictionary in the precise 'question_order'
    ordered_values = [
        result_dict.get(q, np.nan)  # Use .get() and np.nan for missing keys
        for q in question_order
    ]
    # Add the column to the DataFrame, rounding to 3 decimals
    sig_df[column_name] = np.round(ordered_values, 3)

In [434]:
print(model_name)
sig_df

gpt4.1


,Question,Replicate,Reformulated - Replicate,Reverse-coded - Replicate,Priming - Replicate,Preamble - Replicate
0,Climate Change,0.058,-0.032,0.283,0.153,0.023
1,Current Economy,0.232,-0.060,NaN,0.048,-0.011
2,Refugee Allowing,0.113,-0.086,-0.064,-0.004,0.136
3,Health Insurance,0.045,0.026,NaN,0.034,0.162
4,Income Inequality,0.044,0.060,-0.040,0.042,0.046
5,Race Diversity,0.254,-0.068,-0.140,0.013,-0.006
6,Drug Addiction,0.166,-0.022,NaN,0.007,0.007
7,Gay Marriage,0.041,-0.009,0.054,-0.012,0.019
8,Gun Regulation,0.003,0.012,NaN,0.011,0.010
9,Gender Role,0.077,-0.067,0.770,0.106,0.063


In [ ]:
# file_path = f'tables/{model_name}-jsd-diff-t=0.csv'
# sig_df.to_csv(file_path, index=False)

In [457]:
def jsd_divergence_for_bootstrapping(human_sample, bootstrap_sample, base=2):
    p_counts = human_sample
    q_counts = bootstrap_sample

    p = np.array([p_counts.get(k, 0) for k in p_counts], dtype=float)
    q = np.array([q_counts.get(k, 0) for k in p_counts], dtype=float)

    p /= p.sum()
    q /= q.sum()

    epsilon = 1e-10
    p = np.clip(p, epsilon, 1)
    q = np.clip(q, epsilon, 1)

    m = 0.5 * (p + q)
    jsd_pq = 0.5 * (entropy(p, m, base=base) + entropy(q, m, base=base))

    return jsd_pq


### CIs (Human vs conditions)

In [ ]:
human_df = pd.read_csv("../data/2020 ANES_test.csv")

def bootstrap_jsd(path, qid, n_boot=2000, base=2, return_samples=False):

    question_df = pd.read_csv(path + qid + ".csv")
    question_df['Response'] = pd.to_numeric(question_df['Response'], errors='coerce')

    responses = np.array(question_df['Response'].dropna())

    human_responses = {list(d.keys())[0]: list(d.values())[0] for d in human_results}[name_dict[qid]]

    jsd_samples = []

    for _ in range(n_boot):
        sample = np.random.choice(responses, size=len(responses), replace=True) #sample with replacement
        
        counts = Counter(sample)

        jsd_val = jsd_divergence_for_bootstrapping(human_responses, counts, base=base) #compute JSD for this iteration
        jsd_samples.append(jsd_val) #append to the samples list

    jsd_samples = np.array(jsd_samples)

    if return_samples:
        return jsd_samples  # <— raw samples for delta CI

    mean_jsd = jsd_samples.mean()
    ci_lower, ci_upper = np.percentile(jsd_samples, [2.5, 97.5])
    
    return (("mean", mean_jsd), ("ci_lower", ci_lower), ("ci_upper", ci_upper))

def bootstrap_jsd_for_reverse_coded(path, qid, n_boot=2000, base=2, return_samples=False):

    question_df = pd.read_csv(path + qid + ".csv")
    question_df['Response'] = pd.to_numeric(question_df['Response'], errors='coerce')

    reverse_mappings = {
        "Gay Marriage": {1: 3, 3: 1},
        "Refugee Allowing": {1: 2, 2: 1},
        "Income Inequality": {1: 2, 2: 1},
        "Gender Role": {1: 2, 2: 1},
        "Climate Change": {1: 5, 2: 4, 4: 2, 5: 1},
        "Race Diversity": {1: 2, 2: 1}
    }

    qname = name_dict[qid]
    responses = np.array(question_df['Response'].dropna())

    #apply the map to swap the answers
    if qname in reverse_mappings:
        mapping = reverse_mappings[qname]
        responses = np.array([mapping.get(val, val) for val in responses])

    human_responses = {list(d.keys())[0]: list(d.values())[0] for d in human_results}[qname]

    jsd_samples = []

    for _ in range(n_boot):
        sample = np.random.choice(responses, size=len(responses), replace=True)
        counts = Counter(sample)
        jsd_val = jsd_divergence_for_bootstrapping(human_responses, counts, base=base)
        jsd_samples.append(jsd_val)

    jsd_samples = np.array(jsd_samples)

    if return_samples:
        return jsd_samples

    mean_jsd = jsd_samples.mean()
    ci_lower, ci_upper = np.percentile(jsd_samples, [2.5, 97.5])

    return (("mean", mean_jsd), ("ci_lower", ci_lower), ("ci_upper", ci_upper))


/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_57581/2968426189.py:1: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  human_df = pd.read_csv("data/2020 ANES_test.csv")


In [459]:
#Create CIs (human vs conditions) and compare them
ci_replicate = []
ci_reformulated = []
ci_preamble = []
ci_priming = []
ci_reverse_coded = []

for qid in qids:
    ci_replicate.append((name_dict[qid], bootstrap_jsd(path = replicate_path, qid = qid)))
    ci_reformulated.append((name_dict[qid], bootstrap_jsd(path = reformulated_path, qid = qid)))
    ci_priming.append((name_dict[qid], bootstrap_jsd(path = priming_path, qid = qid)))
    ci_preamble.append((name_dict[qid], bootstrap_jsd(path = preamble_path, qid = qid)))

ci_replicate = [{item[0] : item[1]} for item in ci_replicate]
ci_reformulated = [{item[0] : item[1]} for item in ci_reformulated]
ci_priming = [{item[0] : item[1]} for item in ci_priming]
ci_preamble = [{item[0] : item[1]} for item in ci_preamble]

ci_replicate = {list(d.keys())[0]: list(d.values())[0] for d in ci_replicate}
ci_priming = {list(d.keys())[0]: list(d.values())[0] for d in ci_priming}
ci_reformulated = {list(d.keys())[0]: list(d.values())[0] for d in ci_reformulated}
ci_preamble = {list(d.keys())[0]: list(d.values())[0] for d in ci_preamble}


ci_reverse_coded = []

for qid in qids_reverse_coded:
    ci_reverse_coded.append((name_dict[qid], bootstrap_jsd_for_reverse_coded(path = reversed_path, qid = qid)))

ci_reverse_coded = [{item[0] : item[1]} for item in ci_reverse_coded]
ci_reverse_coded = {list(d.keys())[0]: list(d.values())[0] for d in ci_reverse_coded}

In [ ]:
# Mean JS divergence and 95% bootstrap confidence intervals between ANSE and silicon distributions by item and condition.
def format_ci_data(ci_tuple):
    # Convert the list of tuples structure into a dictionary for easy access
    if ci_tuple is None:
        return "N/A"
    data = dict(ci_tuple)
    
    mean = data['mean']
    ci_lower = data['ci_lower']
    ci_upper = data['ci_upper']

    mean_str = f"{mean:.4f}"
    lower_str = f"{ci_lower:4f}"
    upper_str = f"{ci_upper:.4f}"

    return f"{mean_str} [{lower_str}, {upper_str}]"

results_mapping = [
    ('Replicate', ci_replicate),
    ('Reformulated', ci_reformulated),
    ('Reverse-coded', ci_reverse_coded),
    ('Priming', ci_priming),
    ('Preamble', ci_preamble),
]
question_order = ['Climate Change','Current Economy','Refugee Allowing','Health Insurance','Income Inequality','Race Diversity','Drug Addiction','Gay Marriage','Gun Regulation','Gender Role']

ci_df = pd.DataFrame({'Question': question_order})
for column_name, result_dict in results_mapping:
    ordered_formatted_values = [
        format_ci_data(result_dict.get(q,None))
        for q in question_order
    ]

    ci_df[column_name] = ordered_formatted_values


In [461]:
ci_df

,Question,Replicate,Reformulated,Reverse-coded,Priming,Preamble
0,Climate Change,"0.4104 [0.405343, 0.4154]","0.3621 [0.357018, 0.3672]","0.2129 [0.207436, 0.2187]","0.4164 [0.415082, 0.4177]","0.4117 [0.407763, 0.4154]"
1,Current Economy,"0.3091 [0.304436, 0.3138]","0.1202 [0.117036, 0.1233]",N/A,"0.3664 [0.360180, 0.3730]","0.2932 [0.287865, 0.2987]"
2,Refugee Allowing,"0.2755 [0.273477, 0.2772]","0.2103 [0.207052, 0.2137]","0.3077 [0.300673, 0.3148]","0.2756 [0.273477, 0.2772]","0.2499 [0.245174, 0.2546]"
3,Health Insurance,"0.2627 [0.259671, 0.2651]","0.2137 [0.210173, 0.2172]",N/A,"0.2652 [0.263664, 0.2660]","0.2650 [0.263285, 0.2660]"
4,Income Inequality,"0.1976 [0.189069, 0.2060]","0.1490 [0.144714, 0.1532]","0.0634 [0.057961, 0.0687]","0.2716 [0.263886, 0.2792]","0.2143 [0.205844, 0.2231]"
5,Race Diversity,"0.1866 [0.179512, 0.1939]","0.1401 [0.133291, 0.1473]","0.2543 [0.249360, 0.2589]","0.0327 [0.030640, 0.0346]","0.0101 [0.007938, 0.0124]"
6,Drug Addiction,"0.1727 [0.172733, 0.1727]","0.1602 [0.155940, 0.1644]",N/A,"0.1727 [0.172733, 0.1727]","0.1681 [0.164958, 0.1709]"
7,Gay Marriage,"0.1432 [0.138852, 0.1477]","0.0528 [0.048516, 0.0575]","0.1721 [0.172098, 0.1721]","0.1721 [0.172098, 0.1721]","0.1721 [0.172098, 0.1721]"
8,Gun Regulation,"0.1195 [0.112711, 0.1264]","0.0641 [0.058704, 0.0697]",N/A,"0.2157 [0.207106, 0.2248]","0.2044 [0.195714, 0.2135]"
9,Gender Role,"0.0161 [0.013336, 0.0191]","0.1586 [0.150297, 0.1669]","0.8096 [0.801178, 0.8182]","0.1887 [0.183938, 0.1936]","0.1557 [0.148931, 0.1624]"


In [ ]:
# file_path = f'tables/{model_name}-jsd-ci-t=0.csv'
# ci_df.to_csv(file_path, index=False)